In [1]:
# ================================
# 1. IMPORT LIBRARIES
# ================================
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             confusion_matrix)
import lightgbm as lgb
from tqdm import tqdm
import joblib, json, IPython


# ================================
# 2. DATASET PATH
# ================================
dataset_path = "/kaggle/input/datasets/pravallikann/combined-anemia/Fingernails"


# ================================
# 3. IMAGE PREPROCESSING FUNCTION
# ================================
def preprocess_image(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return None

    # BGR → RGB
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Threshold to remove dark background
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    _, thresh = cv2.threshold(gray, 15, 255, cv2.THRESH_BINARY)

    # Find largest contour (fingernail ROI)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(contours) == 0:
        return None

    largest_contour = max(contours, key=cv2.contourArea)
    x, y, w, h      = cv2.boundingRect(largest_contour)
    roi              = img_rgb[y:y+h, x:x+w]

    # Resize to 224x224
    roi = cv2.resize(roi, (224, 224))

    # Normalize (0-1)
    roi = roi / 255.0

    return roi


# ================================
# 4. FEATURE EXTRACTION
#    (CLAHE + HSV + LBP + GLCM + Histogram)
# ================================
def extract_features(img):
    img_255 = (img * 255).astype(np.uint8)

    # ========================
    # A) RGB STATISTICS
    # ========================
    mean_r = np.mean(img[:, :, 0])
    mean_g = np.mean(img[:, :, 1])
    mean_b = np.mean(img[:, :, 2])
    std_r  = np.std(img[:, :, 0])
    std_g  = np.std(img[:, :, 1])
    std_b  = np.std(img[:, :, 2])

    # ========================
    # B) CLAHE ON LAB CHANNEL
    # ========================
    lab     = cv2.cvtColor(img_255, cv2.COLOR_RGB2LAB)
    l, a, b_ch = cv2.split(lab)

    clahe   = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l_clahe = clahe.apply(l)

    mean_l     = np.mean(l_clahe)
    mean_a     = np.mean(a)
    mean_lab_b = np.mean(b_ch)
    std_l      = np.std(l_clahe)
    std_a      = np.std(a)
    std_lab_b  = np.std(b_ch)

    # ========================
    # C) HSV STATISTICS
    # ========================
    hsv    = cv2.cvtColor(img_255, cv2.COLOR_RGB2HSV)
    h, s, v = cv2.split(hsv)

    mean_h = np.mean(h)
    mean_s = np.mean(s)
    mean_v = np.mean(v)
    std_h  = np.std(h)
    std_s  = np.std(s)
    std_v  = np.std(v)

    # ========================
    # D) RED RATIO & PALLOR INDEX
    # ========================
    red_ratio    = mean_r / (mean_r + mean_g + mean_b + 1e-6)
    pallor_index = (mean_r - mean_g) / (mean_r + mean_g + 1e-6)

    # ========================
    # E) RGB HISTOGRAM (16 bins)
    # ========================
    hist_r = cv2.calcHist([img_255], [0], None, [16], [0, 256]).flatten()
    hist_g = cv2.calcHist([img_255], [1], None, [16], [0, 256]).flatten()
    hist_b = cv2.calcHist([img_255], [2], None, [16], [0, 256]).flatten()
    hist_r = hist_r / (np.sum(hist_r) + 1e-6)
    hist_g = hist_g / (np.sum(hist_g) + 1e-6)
    hist_b = hist_b / (np.sum(hist_b) + 1e-6)

    # ========================
    # F) HSV HISTOGRAM (16 bins)
    # ========================
    hist_h = cv2.calcHist([hsv], [0], None, [16], [0, 180]).flatten()
    hist_s = cv2.calcHist([hsv], [1], None, [16], [0, 256]).flatten()
    hist_v = cv2.calcHist([hsv], [2], None, [16], [0, 256]).flatten()
    hist_h = hist_h / (np.sum(hist_h) + 1e-6)
    hist_s = hist_s / (np.sum(hist_s) + 1e-6)
    hist_v = hist_v / (np.sum(hist_v) + 1e-6)

    # ========================
    # G) LBP TEXTURE FEATURE
    # ========================
    def compute_lbp(gray_img, radius=1, n_points=8):
        rows, cols = gray_img.shape
        lbp = np.zeros((rows, cols), dtype=np.uint8)
        for i in range(radius, rows - radius):
            for j in range(radius, cols - radius):
                center = gray_img[i, j]
                binary = 0
                for k in range(n_points):
                    angle = 2 * np.pi * k / n_points
                    nx    = int(round(i + radius * np.sin(angle)))
                    ny    = int(round(j + radius * np.cos(angle)))
                    binary |= (1 << k) if gray_img[nx, ny] >= center else 0
                lbp[i, j] = binary
        return lbp

    gray_img   = cv2.cvtColor(img_255, cv2.COLOR_RGB2GRAY)
    gray_small = cv2.resize(gray_img, (32, 32))
    lbp        = compute_lbp(gray_small)
    lbp_hist   = np.histogram(lbp, bins=16, range=(0, 256))[0]
    lbp_hist   = lbp_hist / (np.sum(lbp_hist) + 1e-6)

    # ========================
    # H) GLCM TEXTURE FEATURES
    # ========================
    def compute_glcm_features(gray_img):
        gray_small = cv2.resize(gray_img, (64, 64))
        gray_small = (gray_small / 16).astype(np.uint8)
        levels     = 16
        glcm       = np.zeros((levels, levels), dtype=np.float64)

        for i in range(gray_small.shape[0]):
            for j in range(gray_small.shape[1] - 1):
                glcm[gray_small[i, j], gray_small[i, j+1]] += 1

        glcm /= (glcm.sum() + 1e-6)

        idxs          = np.arange(levels)
        i_idx, j_idx  = np.meshgrid(idxs, idxs, indexing='ij')

        contrast    = np.sum((i_idx - j_idx) ** 2 * glcm)
        homogeneity = np.sum(glcm / (1 + np.abs(i_idx - j_idx)))
        energy      = np.sum(glcm ** 2)

        mu_i  = np.sum(i_idx * glcm)
        mu_j  = np.sum(j_idx * glcm)
        std_i = np.sqrt(np.sum((i_idx - mu_i) ** 2 * glcm) + 1e-6)
        std_j = np.sqrt(np.sum((j_idx - mu_j) ** 2 * glcm) + 1e-6)
        correlation = np.sum((i_idx - mu_i) * (j_idx - mu_j) * glcm) / (std_i * std_j)

        return [contrast, homogeneity, energy, correlation]

    glcm_features = compute_glcm_features(gray_img)

    # ========================
    # FINAL FEATURE VECTOR
    # ========================
    features = [
        mean_r, mean_g, mean_b, std_r, std_g, std_b,
        mean_l, mean_a, mean_lab_b, std_l, std_a, std_lab_b,
        mean_h, mean_s, mean_v, std_h, std_s, std_v,
        red_ratio, pallor_index,
        *glcm_features
    ]

    features.extend(hist_r)
    features.extend(hist_g)
    features.extend(hist_b)
    features.extend(hist_h)
    features.extend(hist_s)
    features.extend(hist_v)
    features.extend(lbp_hist)

    return features


# ================================
# 5. LOAD DATASET & BUILD FEATURE MATRIX
# ================================
image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}

def find_class_folders(root_path):
    class_folders = {}
    for dirpath, dirnames, filenames in os.walk(root_path):
        images = [f for f in filenames if os.path.splitext(f)[1].lower() in image_extensions]
        if images:
            folder_name = os.path.basename(dirpath)
            class_folders[folder_name] = dirpath
    return class_folders

found = find_class_folders(dataset_path)
print("Auto-detected class folders:")
for name, path in found.items():
    print(f"  '{name}' → {path}")

X = []
y = []

# ⚠️ Update keys to match your actual folder names
class_to_label = {
    "Anemic":    0,
    "NonAnemic": 1
}
classes = list(class_to_label.keys())

for class_name, label in class_to_label.items():
    if class_name not in found:
        print(f"WARNING: '{class_name}' not found! Available: {list(found.keys())}")
        continue
    class_folder = found[class_name]
    for img_name in tqdm(os.listdir(class_folder), desc=f"Loading {class_name}"):
        img_path      = os.path.join(class_folder, img_name)
        processed_img = preprocess_image(img_path)
        if processed_img is not None:
            features = extract_features(processed_img)
            X.append(features)
            y.append(label)

X = np.array(X)
y = np.array(y)

print(f"\nFeature Matrix Shape : {X.shape}")
print(f"Labels Shape         : {y.shape}")


# ================================
# 6. TRAIN-TEST SPLIT (STRATIFIED 80-20)
# ================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f"Train samples : {len(X_train)}")
print(f"Test  samples : {len(X_test)}")


# ================================
# 7. BUILD LIGHTGBM DATASET
# ================================
train_data = lgb.Dataset(X_train, label=y_train)
val_data   = lgb.Dataset(X_test,  label=y_test, reference=train_data)


# ================================
# 8. LIGHTGBM PARAMETERS
# ================================
# Class weight for imbalance
class_counts    = np.bincount(y_train)
scale_pos_weight = class_counts[0] / class_counts[1]

params = {
    'objective'        : 'binary',
    'metric'           : ['binary_logloss', 'auc'],
    'boosting_type'    : 'gbdt',
    'num_leaves'       : 63,
    'max_depth'        : -1,
    'learning_rate'    : 0.05,
    'n_estimators'     : 500,
    'subsample'        : 0.8,
    'colsample_bytree' : 0.8,
    'min_child_samples': 20,
    'reg_alpha'        : 0.1,
    'reg_lambda'       : 0.1,
    'scale_pos_weight' : scale_pos_weight,   # handles class imbalance
    'random_state'     : 42,
    'verbose'          : -1
}

print("\nLightGBM Parameters:")
for k, v in params.items():
    print(f"  {k:25s}: {v}")


# ================================
# 9. TRAIN LIGHTGBM MODEL
# ================================
print("\nTraining LightGBM...")

callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=True),
    lgb.log_evaluation(period=50)
]

model = lgb.train(
    params,
    train_data,
    num_boost_round=500,
    valid_sets=[train_data, val_data],
    valid_names=['train', 'val'],
    callbacks=callbacks
)

print("\nTraining complete!")
print(f"Best iteration : {model.best_iteration}")


# ================================
# 10. SAVE TRAINED MODEL
# ================================
save_path = "/kaggle/working/anemia_lightgbm_model.pkl"
joblib.dump(model, save_path)
print(f"\nModel saved at: {save_path}")


# ================================
# 11. PREDICTION
# ================================
y_prob = model.predict(X_test, num_iteration=model.best_iteration)
y_pred = (y_prob >= 0.5).astype(int)


# ================================
# 12. EVALUATION METRICS
# ================================
accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)
roc_auc   = roc_auc_score(y_test, y_prob)

print("\n===== MODEL PERFORMANCE =====")
print("Accuracy  :", round(accuracy,  4))
print("Precision :", round(precision, 4))
print("Recall    :", round(recall,    4))
print("F1 Score  :", round(f1,        4))
print("ROC-AUC   :", round(roc_auc,   4))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=classes))


# ================================
# 13. CONFUSION MATRIX
# ================================
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.colorbar(im, ax=ax)
ax.set_xticks([0, 1]); ax.set_xticklabels(classes)
ax.set_yticks([0, 1]); ax.set_yticklabels(classes)
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
ax.set_title("LightGBM — Confusion Matrix")

for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=14)

plt.tight_layout()
plt.savefig("/kaggle/working/lightgbm_confusion_matrix.png", dpi=150)
plt.show()


# ================================
# 14. FEATURE IMPORTANCE PLOT
# ================================
lgb.plot_importance(model, max_num_features=20, figsize=(10, 7),
                    title="LightGBM — Top 20 Feature Importances")
plt.tight_layout()
plt.savefig("/kaggle/working/lightgbm_feature_importance.png", dpi=150)
plt.show()


# ================================
# 15. LOAD SAVED MODEL
# ================================
load_path    = "/kaggle/working/anemia_lightgbm_model.pkl"
loaded_model = joblib.load(load_path)
print("Model loaded successfully!")


# ================================
# 16. REAL IMAGE PREDICTION
# ================================
def predict_anemia(image_path, model):
    processed_img = preprocess_image(image_path)
    if processed_img is None:
        print("Image processing failed.")
        return

    features = extract_features(processed_img)
    features = np.array(features).reshape(1, -1)

    # LightGBM predict returns probability of class 1 (NonAnemic)
    prob_nonanemic = model.predict(features, num_iteration=model.best_iteration)[0]
    prob_anemic    = 1 - prob_nonanemic
    prediction     = 1 if prob_nonanemic >= 0.5 else 0

    if prediction == 0:
        result      = "Anemic"
        probability = prob_anemic
    else:
        result      = "Non-Anemic"
        probability = prob_nonanemic

    print("\nPrediction              :", result)
    print(f"Probability ({result}) :", round(probability, 4))


# ================================
# PREDICT ON REAL IMAGE
# ================================
predict_anemia("/kaggle/input/datasets/pravallikann/combined-anemia/Fingernails/NonAnemic/Image_012.png", loaded_model)  # ← change path


# ================================
# 17. SAVE NOTEBOOK
# ================================
ip   = IPython.get_ipython()
hist = list(ip.history_manager.get_range(output=False))

notebook = {
    "nbformat": 4,
    "nbformat_minor": 5,
    "metadata": {
        "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
        "language_info": {"name": "python", "version": "3.12.0"}
    },
    "cells": [
        {
            "cell_type"       : "code",
            "execution_count" : line_num,
            "metadata"        : {},
            "outputs"         : [],
            "source"          : src
        }
        for _, line_num, src in hist if src.strip()
    ]
}

nb_save_path = "/kaggle/working/anemia_lightgbm_notebook.ipynb"
with open(nb_save_path, "w", encoding="utf-8") as f:
    json.dump(notebook, f, indent=2, ensure_ascii=False)

print(f"✅ Notebook saved at : {nb_save_path}")
print(f"   Total cells       : {len(notebook['cells'])}")
print(f"   File size         : {round(os.path.getsize(nb_save_path)/1024, 2)} KB")
print("\n👉 Output tab → Download anemia_lightgbm_notebook.ipynb")